In [ ]:
"""
ECHR Application Form Parser
===========================

Extracts structured data from a printed/scanned European Court of
Human Rights (ECHR) application form PDF using OCR (Tesseract).

Key characteristics:
- Works with image-only (printed/scanned) PDFs
- Assumes the ECHR form layout is fixed
- Always returns a canonical nested dictionary with all fields present
"""

from __future__ import annotations

import re
from typing import List, Dict, Optional
import pytesseract
 
import pdfplumber
from PIL import Image


# ==================================================================
# OCR CONFIGURATION
# ==================================================================

# Update this path if Tesseract is installed elsewhere
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)


# ==================================================================
# OCR & TEXT NORMALIZATION
# ==================================================================

def clean_line(line: str) -> str:
    """
    Normalize OCR noise and common checkbox / layout artefacts.
    """
    return (
        line.replace("©)", "")
            .replace("@)", "")
            .replace("|", "")
            .strip()
    )


def extract_lines_from_pdf(pdf_path: str) -> List[str]:
    """
    Perform OCR on every page of the PDF and return a list
    of non-empty text lines in reading order.
    """
    lines: List[str] = []

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            image: Image.Image = page.to_image(resolution=300).original
            text: str = pytesseract.image_to_string(image, lang="eng")

            for raw_line in text.splitlines():
                line = raw_line.strip()
                if line:
                    lines.append(line)

    return lines


# ==================================================================
# GENERIC PARSING HELPERS
# ==================================================================

def value_after(
    lines: List[str],
    label: str,
    stop_regex: str = r"^\d+\.|^[A-Z]\."
) -> Optional[str]:
    """
    Extract text immediately following a labeled field until another
    numbered field or section heading appears.
    """
    for index, line in enumerate(lines):
        if label in line:
            collected: List[str] = []

            for next_line in lines[index + 1 : index + 8]:
                if re.search(stop_regex, next_line):
                    break
                collected.append(clean_line(next_line))

            return " ".join(collected).strip()

    return None


def extract_section_block(
    lines: List[str],
    start_marker: str,
    end_marker: str
) -> Optional[str]:
    """
    Extract a long narrative section between two textual markers.
    """
    captured: List[str] = []
    capturing = False

    for line in lines:
        if line.startswith(start_marker):
            capturing = True
            continue

        if capturing and line.startswith(end_marker):
            break

        if capturing:
            captured.append(clean_line(line))

    return " ".join(captured).strip() or None


# ==================================================================
# CANONICAL FORM TEMPLATE
# ==================================================================

def echr_form_template() -> Dict[str, object]:
    """
    Canonical template of the ECHR application form.
    All fields exist by default; values are filled when detected.
    """
    return {
        "A. Applicant": {
            "A.1 Individual": {
                "Surname": None,
                "First name(s)": None,
                "Date of birth": None,
                "Place of birth": None,
                "Nationality": None,
                "Address": None,
                "Telephone": None,
                "Email": None,
                "Sex": None,
            },
            "A.2 Organisation": {
                "Name": None,
                "Identification number": None,
                "Date of registration": None,
                "Activity": None,
                "Registered address": None,
                "Telephone": None,
                "Email": None,
            },
        },

        "B. States": {
            "Selected": [],
            "Available": [
                "Albania", "Andorra", "Armenia", "Austria", "Azerbaijan",
                "Belgium", "Bosnia and Herzegovina", "Bulgaria", "Croatia",
                "Cyprus", "Czech Republic", "Denmark", "Estonia", "Finland",
                "France", "Georgia", "Germany", "Greece", "Hungary", "Iceland",
                "Ireland", "Italy", "Latvia", "Liechtenstein", "Lithuania",
                "Luxembourg", "Malta", "Moldova", "Monaco", "Montenegro",
                "Netherlands", "North Macedonia", "Norway", "Poland",
                "Portugal", "Romania", "San Marino", "Serbia", "Slovakia",
                "Slovenia", "Spain", "Sweden", "Switzerland", "Türkiye",
                "Ukraine", "United Kingdom",
            ],
        },

        "C. Representative (Individual applicant)": {
            "C.1 Non-lawyer": {
                "Capacity / relationship": None,
                "Surname": None,
                "First name(s)": None,
                "Nationality": None,
                "Address": None,
                "Telephone": None,
                "Fax": None,
                "Email": None,
            },
            "C.2 Lawyer": {
                "Surname": None,
                "First name(s)": None,
                "Nationality": None,
                "Address": None,
                "Telephone": None,
                "Fax": None,
                "Email": None,
                "eComms email": None,
            },
        },

        "D. Representative (Organisation)": {
            "D.1 Organisation official": {
                "Capacity / function": None,
                "Surname": None,
                "First name(s)": None,
                "Nationality": None,
                "Address": None,
                "Telephone": None,
                "Fax": None,
                "Email": None,
            },
            "D.2 Lawyer": {
                "Surname": None,
                "First name(s)": None,
                "Nationality": None,
                "Address": None,
                "Telephone": None,
                "Fax": None,
                "Email": None,
            },
        },

        "E. Statement of facts": None,

        "F. Alleged violations": {
            "Article 61": None,
            "Article 62": None,
        },

        "G. Admissibility": {
            "Complaints summary": None,
            "Unused remedies": None,
            "Why unused": None,
        },

        "H. Other international proceedings": {
            "Any proceedings": None,
            "Details": None,
            "Other ECHR applications": None,
            "Application numbers": None,
        },

        "I. List of accompanying documents": [],

        "J. Final declaration": {
            "Additional comments": None,
            "Date": None,
            "Signed by": None,
            "Correspondence contact": None,
        },
    }


# ==================================================================
# TEMPLATE POPULATION
# ==================================================================

def fill_template(template: Dict[str, object], lines: List[str]) -> Dict[str, object]:
    """
    Populate the canonical template using OCR-extracted text lines.
    """

    # --- A.1 Individual applicant ---
    individual = template["A. Applicant"]["A.1 Individual"]

    individual["Surname"] = value_after(lines, "1. Surname")
    individual["First name(s)"] = value_after(lines, "2. First name")
    individual["Date of birth"] = value_after(lines, "3. Date of birth")
    individual["Place of birth"] = value_after(lines, "4. Place of birth")
    individual["Nationality"] = value_after(lines, "5. Nationality")
    individual["Address"] = value_after(lines, "6. Address")
    individual["Telephone"] = value_after(lines, "7. Telephone")
    individual["Email"] = value_after(lines, "8. Email")

    for line in lines:
        lower = line.lower()
        if "female" in lower:
            individual["Sex"] = "female"
        elif "male" in lower:
            individual["Sex"] = "male"

    # --- B. States ---
    if any("GEO - Georgia" in l for l in lines):
        template["B. States"]["Selected"].append("Georgia")

    # --- C.2 Lawyer ---
    lawyer = template["C. Representative (Individual applicant)"]["C.2 Lawyer"]

    lawyer["Surname"] = value_after(lines, "26. Surname")
    lawyer["First name(s)"] = value_after(lines, "27. First name")
    lawyer["Nationality"] = value_after(lines, "28. Nationality")
    lawyer["Telephone"] = value_after(lines, "30. Telephone")
    lawyer["Email"] = value_after(lines, "32. Email")
    lawyer["eComms email"] = value_after(lines, "37. Email")

    # --- E. Statement of facts ---
    template["E. Statement of facts"] = extract_section_block(
        lines,
        start_marker="E. Statement of the facts",
        end_marker="European Court of Human Rights - Application form 6",
    )

    # --- F. Alleged violations ---
    template["F. Alleged violations"]["Article 61"] = value_after(
        lines,
        "material aspect of Article 2",
        stop_regex=r"procedural|62\.",
    )

    template["F. Alleged violations"]["Article 62"] = value_after(
        lines,
        "procedural aspect of Article 2",
        stop_regex=r"G\.",
    )

    # --- G. Admissibility ---
    template["G. Admissibility"]["Unused remedies"] = (
        "Yes" if any("©) Yes" in l for l in lines) else "No"
    )

    return template


# ==================================================================
# PUBLIC API
# ==================================================================

def parse_echr_application(pdf_path: str) -> Dict[str, object]:
    """
    Single public entry point.

    Given a printed / scanned ECHR application PDF (filled or blank),
    return a fully populated canonical nested dictionary.
    """
    lines = extract_lines_from_pdf(pdf_path)
    template = echr_form_template()
    return fill_template(template, lines)


In [2]:
# formated_data_dict= parse_echr_application("data/printed Filled ECHR application form.pdf")

In [ ]:

import json
from pathlib import Path

formated_data_dict = parse_echr_application("data/printed Filled ECHR application form.pdf")

folder = Path("app data")
folder.mkdir(exist_ok=True)

file_path = folder / "data.json"

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(formated_data_dict, f, indent=2)
